# TrackLink USBL — Position Visualisation

Plots ship track, resolved USBL target positions, and re-navigation camera
poses on an interactive map for a single deployment.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
DEPLOYMENT: str = "qdch0ftq_20120430_002423"

# NOTE: Deployments - QDCH0FTQ
# qdch0ftq_20100428_020202
# qdch0ftq_20110415_020103
# qdch0ftq_20120430_002423
# qdch0ftq_20130406_023610

# Note: Deployments - Site QDCHDMY1
# qdchdmy1_20110416_005411
# qdchdmy1_20120501_071203
# qdchdmy1_20130406_081713

USBL_DIR: Path = Path("/data/exos_01/acfr_messages_v4_telemetry_processed")
RENAV_DIR: Path = Path("/data/exos_01/acfr_camera_poses_renav")

USBL_FILE: Path = USBL_DIR / f"{DEPLOYMENT}_usbl_tracklink.csv"
RENAV_FILE: Path = RENAV_DIR / f"{DEPLOYMENT}_cameras.csv"

## Load data

In [ ]:
USBL_REQUIRED_COLS: list[str] = [
    "timestamp",
    "ship_latitude",
    "ship_longitude",
    "ship_heading",
    "ship_roll",
    "ship_pitch",
    "target_bearing",
    "target_slant_range",
    "target_depth",
    "target_latitude",
    "target_longitude",
    "horizontal_range",
    "position_uncertainty",
]

usbl: pd.DataFrame = pd.read_csv(
    USBL_FILE, parse_dates=["timestamp"], date_format="ISO8601"
)

missing = [col for col in USBL_REQUIRED_COLS if col not in usbl.columns]
if missing:
    raise ValueError(f"USBL file is missing required columns: {missing}")

usbl = gpd.GeoDataFrame(
    usbl,
    geometry=gpd.points_from_xy(
        usbl["target_longitude"], usbl["target_latitude"]
    ),
    crs="EPSG:4326",
)

print(f"USBL rows: {len(usbl)}")
usbl.head(3)

In [ ]:
renav: pd.DataFrame = pd.read_csv(RENAV_FILE)
renav["datetime"] = pd.to_datetime(renav["timestamp"], unit="s", utc=True)

renav = gpd.GeoDataFrame(
    renav,
    geometry=gpd.points_from_xy(renav["longitude"], renav["latitude"]),
    crs="EPSG:4326",
)

print(f"Renav rows: {len(renav)}")
renav.head(3)

## Overview map: ship track and USBL target positions

In [ ]:
center_lat: float = float(usbl.geometry.y.mean())
center_lon: float = float(usbl.geometry.x.mean())

# Elapsed time in minutes from the earliest timestamp across both datasets.
# Computed here so subsequent cells can reuse these arrays.
usbl_t_s: np.ndarray = (usbl["timestamp"].astype(np.int64) / 1e9).to_numpy()
renav_t_s: np.ndarray = (renav["datetime"].astype(np.int64) / 1e9).to_numpy()


t0: float = min(float(usbl_t_s.min()), float(renav_t_s.min()))
usbl_elapsed: np.ndarray = (usbl_t_s - t0) / 60.0
renav_elapsed: np.ndarray = (renav_t_s - t0) / 60.0
t_max: float = max(float(usbl_elapsed.max()), float(renav_elapsed.max()))

colorscale: str = "Viridis"

fig = go.Figure()

# Ship track — line kept fixed-colour; markers carry the time gradient
fig.add_trace(
    go.Scattermapbox(
        lat=usbl["ship_latitude"],
        lon=usbl["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=4,
            color=usbl_elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=t_max,
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

# USBL target positions
fig.add_trace(
    go.Scattermapbox(
        lat=usbl.geometry.y,
        lon=usbl.geometry.x,
        mode="markers",
        marker=dict(
            size=5,
            color=usbl_elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=t_max,
            showscale=False,
        ),
        name="USBL target",
        customdata=np.stack(
            [
                usbl["target_slant_range"],
                usbl["target_depth"],
                usbl["position_uncertainty"],
            ],
            axis=1,
        ),
        hovertemplate=(
            "<b>USBL target</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Slant range: %{customdata[0]:.1f} m<br>"
            "Depth: %{customdata[1]:.1f} m<br>"
            "Uncertainty: %{customdata[2]:.1f} m<br>"
            "<extra></extra>"
        ),
    )
)

fig.update_layout(
    title=f"Ship track and USBL target positions — {DEPLOYMENT}",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat, lon=center_lon),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

## Comparison map: USBL vs. renav (±30 s overlap window)

Only USBL pings and ship positions within 30 seconds of the renav time window
are shown, so the three sources are directly comparable in time.

In [ ]:
RENAV_WINDOW_MARGIN_S: float = 30.0

# Match all USBL pings to nearest renav sample — reused in the spatial offset cell.
usbl_renav_matched: pd.DataFrame = pd.merge_asof(
    usbl.sort_values("timestamp")[["timestamp", "geometry"]],
    renav.sort_values("datetime")[["datetime", "geometry"]].rename(
        columns={"geometry": "renav_geometry"}
    ),
    left_on="timestamp",
    right_on="datetime",
    direction="nearest",
)

margin: pd.Timedelta = pd.Timedelta(seconds=RENAV_WINDOW_MARGIN_S)
in_window: pd.Series = (
    usbl_renav_matched["timestamp"] >= renav["datetime"].min() - margin
) & (usbl_renav_matched["timestamp"] <= renav["datetime"].max() + margin)

usbl_win: gpd.GeoDataFrame = usbl[in_window.to_numpy()].reset_index(drop=True)
win_matched: pd.DataFrame = usbl_renav_matched[in_window].reset_index(drop=True)
usbl_elapsed_win: np.ndarray = usbl_elapsed[in_window.to_numpy()]

print(
    f"USBL pings in window: {in_window.sum()} / {len(usbl)} "
    f"({100 * in_window.mean():.1f} %)"
)

utm_crs = usbl_win.geometry.estimate_utm_crs()
win_offset_m: pd.Series = (
    gpd.GeoSeries(win_matched["geometry"], crs="EPSG:4326")
    .to_crs(utm_crs)
    .distance(
        gpd.GeoSeries(win_matched["renav_geometry"], crs="EPSG:4326").to_crs(
            utm_crs
        ),
        align=False,
    )
)

center_lat2: float = float(renav.geometry.y.mean())
center_lon2: float = float(renav.geometry.x.mean())

fig_cmp = go.Figure()

# Ship track (filtered to renav window)
fig_cmp.add_trace(
    go.Scattermapbox(
        lat=usbl_win["ship_latitude"],
        lon=usbl_win["ship_longitude"],
        mode="lines+markers",
        marker=dict(
            size=4,
            color=usbl_elapsed_win,
            colorscale=colorscale,
            cmin=0,
            cmax=t_max,
            showscale=True,
            colorbar=dict(title="Elapsed (min)", x=0.92),
        ),
        line=dict(width=1, color="royalblue"),
        name="Ship track",
        hovertemplate=(
            "<b>Ship</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "<extra></extra>"
        ),
    )
)

# USBL target positions (filtered)
fig_cmp.add_trace(
    go.Scattermapbox(
        lat=usbl_win.geometry.y,
        lon=usbl_win.geometry.x,
        mode="markers",
        marker=dict(
            size=5,
            color=usbl_elapsed_win,
            colorscale=colorscale,
            cmin=0,
            cmax=t_max,
            showscale=False,
        ),
        name="USBL target",
        customdata=np.stack(
            [
                usbl_win["target_slant_range"],
                usbl_win["target_depth"],
                usbl_win["position_uncertainty"],
                win_offset_m,
            ],
            axis=1,
        ),
        hovertemplate=(
            "<b>USBL target</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Slant range: %{customdata[0]:.1f} m<br>"
            "Depth: %{customdata[1]:.1f} m<br>"
            "Uncertainty: %{customdata[2]:.1f} m<br>"
            "Offset to renav: %{customdata[3]:.1f} m<br>"
            "<extra></extra>"
        ),
    )
)

# Renav camera poses (all)
fig_cmp.add_trace(
    go.Scattermapbox(
        lat=renav.geometry.y,
        lon=renav.geometry.x,
        mode="markers",
        marker=dict(
            size=4,
            color=renav_elapsed,
            colorscale=colorscale,
            cmin=0,
            cmax=t_max,
            showscale=False,
            opacity=0.6,
        ),
        name="Renav cameras",
        customdata=np.stack(
            [renav["depth"], renav["altitude"], renav["heading"]],
            axis=1,
        ),
        hovertemplate=(
            "<b>Renav</b><br>"
            "Lat: %{lat:.6f}<br>"
            "Lon: %{lon:.6f}<br>"
            "Depth: %{customdata[0]:.1f} m<br>"
            "Altitude: %{customdata[1]:.2f} m<br>"
            "Heading: %{customdata[2]:.1f}°<br>"
            "<extra></extra>"
        ),
    )
)

fig_cmp.update_layout(
    title=f"USBL vs. renav — {DEPLOYMENT} (±{RENAV_WINDOW_MARGIN_S:.0f} s window)",
    mapbox=dict(
        style="open-street-map",
        center=dict(lat=center_lat2, lon=center_lon2),
        zoom=14,
    ),
    legend=dict(x=0.01, y=0.99),
    height=700,
    margin=dict(l=0, r=0, t=40, b=0),
)

fig_cmp.show()

## Time series: depth, slant range, and position uncertainty

In [ ]:
fig2 = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        "Interpolated depth (m)",
        "Slant range (m)",
        "Position uncertainty (m)",
    ),
    vertical_spacing=0.07,
)

fig2.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["target_depth"],
        mode="lines",
        name="Depth",
        line=dict(color="steelblue"),
    ),
    row=1,
    col=1,
)

fig2.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["target_slant_range"],
        mode="lines",
        name="Slant range",
        line=dict(color="darkorange"),
    ),
    row=2,
    col=1,
)

fig2.add_trace(
    go.Scatter(
        x=usbl["timestamp"],
        y=usbl["position_uncertainty"],
        mode="lines",
        name="Uncertainty",
        line=dict(color="crimson"),
    ),
    row=3,
    col=1,
)

fig2.update_yaxes(autorange="reversed", row=1, col=1)
fig2.update_layout(
    title=f"USBL time series — {DEPLOYMENT}",
    height=700,
    showlegend=False,
)

fig2.show()

## Spatial offset: USBL vs. renav

Matches each USBL ping to the nearest renav sample by timestamp. Pings with
no renav sample within `MAX_TIME_GAP_S` seconds are excluded (shown as gaps
in the plot). Distance is computed via UTM projection for accuracy.

In [ ]:
MAX_TIME_GAP_S: float = 10.0

within_gap: pd.Series = (
    usbl_renav_matched["timestamp"] - usbl_renav_matched["datetime"]
).abs() <= pd.Timedelta(seconds=MAX_TIME_GAP_S)
print(
    f"USBL pings matched within {MAX_TIME_GAP_S:.0f} s: "
    f"{within_gap.sum()} / {len(usbl)} ({100 * within_gap.mean():.1f} %)"
)

utm_crs = usbl.geometry.estimate_utm_crs()
offset_m: pd.Series = (
    gpd.GeoSeries(usbl_renav_matched["geometry"], crs="EPSG:4326")
    .to_crs(utm_crs)
    .distance(
        gpd.GeoSeries(
            usbl_renav_matched["renav_geometry"], crs="EPSG:4326"
        ).to_crs(utm_crs),
        align=False,
    )
    .where(within_gap)
)

fig3 = go.Figure(
    go.Scatter(
        x=usbl_renav_matched["timestamp"],
        y=offset_m,
        mode="lines",
        name="USBL–renav offset",
        line=dict(color="purple"),
    )
)
fig3.add_hline(
    y=float(offset_m.median()),
    line_dash="dash",
    line_color="grey",
    annotation_text=f"Median {offset_m.median():.1f} m",
)
fig3.update_layout(
    title=f"USBL–renav spatial offset — {DEPLOYMENT}",
    xaxis_title="Time",
    yaxis_title="Offset (m)",
    height=400,
)
fig3.show()